In [ ]:
use std::fs::File;
use std::io::{Read, Write};

// MT19937の逆演算
fn untemper(mut y: u32) -> u32 {
    y ^= y >> 18;
    y ^= (y << 15) & 0xefc60000;
    let mut a = y;
    for _ in 0..4 { a = y ^ ((a << 7) & 0x9d2c5680); }
    y = a;
    let mut b = y;
    for _ in 0..2 { b = y ^ (b >> 11); }
    y = b;
    y
}

struct MT19937 {
    mt: [u32; 624],
    mti: usize,
}

impl MT19937 {
    fn from_outputs(outputs: &[u32]) -> Self {
        let mut mt = [0u32; 624];
        for i in 0..624 { mt[i] = untemper(outputs[i]); }
        MT19937 { mt, mti: 624 }
    }
    fn genrand_int32(&mut self) -> u32 {
        if self.mti >= 624 {
            for kk in 0..227 {
                let y = (self.mt[kk] & 0x80000000) | (self.mt[kk+1] & 0x7fffffff);
                self.mt[kk] = self.mt[kk+397] ^ (y >> 1) ^ (if y & 1 == 1 { 0x9908b0df } else { 0 });
            }
            for kk in 227..623 {
                let y = (self.mt[kk] & 0x80000000) | (self.mt[kk+1] & 0x7fffffff);
                self.mt[kk] = self.mt[kk-227] ^ (y >> 1) ^ (if y & 1 == 1 { 0x9908b0df } else { 0 });
            }
            let y = (self.mt[623] & 0x80000000) | (self.mt[0] & 0x7fffffff);
            self.mt[623] = self.mt[396] ^ (y >> 1) ^ (if y & 1 == 1 { 0x9908b0df } else { 0 });
            self.mti = 0;
        }
        let mut y = self.mt[self.mti];
        self.mti += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c5680;
        y ^= (y << 15) & 0xefc60000;
        y ^= y >> 18;
        y
    }
}

// 1. 提供されたバイナリデータを平文として定義
let cpp_hex = "2F 2F 20 20 67 2B 2B 20 2D 4F 32 20 2D 6F 20 65 6E 63 72 79 70 74 2E 65 78 65 20 65 6E 63 72 79 70 74 2E 63 70 70 20 6D 74 31 39 39 33 37 61 72 2E 63 70 70 0A 0A 23 69 6E 63 6C 75 64 65 20 3C 73 74 64 69 6F 2E 68 3E 0A 23 69 6E 63 6C 75 64 65 20 3C 73 74 64 6C 69 62 2E 68 3E 0A 23 69 6E 63 6C 75 64 65 20 22 6D 74 31 39 39 33 37 61 72 2E 68 22 0A 0A 74 79 70 65 64 65 66 20 75 6E 73 69 67 6E 65 64 20 6C 6F 6E 67 20 64 77 6F 72 64 3B 0A 0A 76 6F 69 64 20 69 6E 69 74 69 61 6C 69 7A 65 28 63 6F 6E 73 74 20 63 68 61 72 20 2A 73 65 65 64 29 3B 0A 76 6F 69 64 20 65 6E 63 72 79 70 74 28 63 6F 6E 73 74 20 63 68 61 72 20 2A 70 6C 61 69 6E 2C 20 63 6F 6E 73 74 20 63 68 61 72 20 2A 63 72 79 70 74 2C 20 63 6F 6E 73 74 20 63 68 61 72 20 2A 6B 65 79 29 3B 0A 76 6F 69 64 20 64 65 63 72 79 70 74 28 63 6F 6E 73 74 20 63 68 61 72 20 2A 70 6C 61 69 6E 2C 20 63 6F 6E 73 74 20 63 68 61 72 20 2A 63 72 79 70 74 2C 20 63 6F 6E 73 74 20 63 68 61 72 20 2A 6B 65 79 29 3B 0A 69 6E 74 20 6D 69 6E 28 69 6E 74 20 61 2C 20 69 6E 74 20 62 29 20 7B 20 72 65 74 75 72 6E 20 61 3C 62 20 3F 20 61 20 3A 20 62 3B 20 7D 0A 0A 69 6E 74 20 6D 61 69 6E 28 29 0A 7B 0A 20 20 20 20 69 6E 69 74 69 61 6C 69 7A 65 28 22 73 65 65 64 22 29 3B 0A 20 20 20 20 0A 20 20 20 20 65 6E 63 72 79 70 74 28 22 65 6E 63 72 79 70 74 2E 63 70 70 22 2C 20 22 65 6E 63 72 79 70 74 2E 65 6E 63 22 2C 20 22 65 6E 63 72 79 70 74 2E 6B 65 79 22 29 3B 0A 20 20 20 20 65 6E 63 72 79 70 74 28 22 66 6C 61 67 2E 6A 70 67 22 2C 20 22 66 6C 61 67 2E 65 6E 63 22 2C 20 22 66 6C 61 67 2E 6B 65 79 22 29 3B 0A 20 20 20 20 0A 20 20 20 20 2F 2F 20 20 64 65 63 72 79 70 74 28 22 65 6E 63 72 79 70 74 5F 64 65 63 2E 63 70 70 22 2C 20 22 65 6E 63 72 79 70 74 2E 65 6E 63 22 2C 20 22 65 6E 63 72 79 70 74 2E 6B 65 79 22 29 3B 0A 20 20 20 20 2F 2F 20 20 64 65 63 72 79 70 74 28 22 66 6C 61 67 5F 64 65 63 2E 6A 70 67 22 2C 20 22 66 6C 61 67 2E 65 6E 63 22 2C 20 22 66 6C 61 67 2E 6B 65 79 22 29 3B 0A 7D 0A 0A 76 6F 69 64 20 69 6E 69 74 69 61 6C 69 7A 65 28 63 6F 6E 73 74 20 63 68 61 72 20 2A 73 65 65 64 29 0A 7B 0A 20 20 20 20 63 6F 6E 73 74 20 69 6E 74 20 4E 20 3D 20 31 30 32 34 3B 0A 20 20 20 20 0A 20 20 20 20 46 49 4C 45 20 2A 66 20 3D 20 66 6F 70 65 6E 28 73 65 65 64 2C 20 22 72 62 22 29 3B 0A 20 20 20 20 69 66 20 28 66 3D 3D 4E 55 4C 4C 29 0A 20 20 20 20 7B 0A 20 20 20 20 20 20 20 20 70 72 69 6E 74 66 28 22 46 61 69 6C 65 64 20 74 6F 20 6F 70 65 6E 20 25 73 5C 6E 22 2C 20 73 65 65 64 29 3B 0A 20 20 20 20 20 20 20 20 65 78 69 74 28 2D 31 29 3B 0A 20 20 20 20 7D 0A 20 20 20 20 0A 20 20 20 20 64 77 6F 72 64 20 62 75 66 5B 4E 5D 3B 0A 20 20 20 20 66 72 65 61 64 28 62 75 66 2C 20 73 69 7A 65 6F 66 20 28 64 77 6F 72 64 29 2C 20 4E 2C 20 66 29 3B 0A 20 20 20 20 0A 20 20 20 20 66 63 6C 6F 73 65 28 66 29 3B 0A 20 20 20 20 0A 20 20 20 20 69 6E 69 74 5F 62 79 5F 61 72 72 61 79 28 62 75 66 2C 20 4E 29 3B 0A 7D 0A 0A 76 6F 69 64 20 65 6E 63 72 79 70 74 28 63 6F 6E 73 74 20 63 68 61 72 20 2A 70 6C 61 69 6E 2C 20 63 6F 6E 73 74 20 63 68 61 72 20 2A 63 72 79 70 74 2C 20 63 6F 6E 73 74 20 63 68 61 72 20 2A 6B 65 79 29 0A 7B 0A 20 20 20 20 46 49 4C 45 20 2A 66 70 20 3D 20 66 6F 70 65 6E 28 70 6C 61 69 6E 2C 20 22 72 62 22 29 3B 0A 20 20 20 20 69 66 20 28 66 70 3D 3D 4E 55 4C 4C 29 0A 20 20 20 20 7B 0A 20 20 20 20 20 20 20 20 70 72 69 6E 74 66 28 22 46 61 69 6C 65 64 20 74 6F 20 6F 70 65 6E 20 25 73 5C 6E 22 2C 20 70 6C 61 69 6E 29 3B 0A 20 20 20 20 20 20 20 20 65 78 69 74 28 2D 31 29 3B 0A 20 20 20 20 7D 0A 20 20 20 20 46 49 4C 45 20 2A 66 63 20 3D 20 66 6F 70 65 6E 28 63 72 79 70 74 2C 20 22 77 62 22 29 3B 0A 20 20 20 20 69 66 20 28 66 63 3D 3D 4E 55 4C 4C 29 0A 20 20 20 20 7B 0A 20 20 20 20 20 20 20 20 70 72 69 6E 74 66 28 22 46 61 69 6C 65 64 20 74 6F 20 6F 70 65 6E 20 25 73 5C 6E 22 2C 20 63 72 79 70 74 29 3B 0A 20 20 20 20 20 20 20 20 65 78 69 74 28 2D 31 29 3B 0A 20 20 20 20 7D 0A 20 20 20 20 46 49 4C 45 20 2A 66 6B 20 3D 20 66 6F 70 65 6E 28 6B 65 79 2C 20 22 77 62 22 29 3B 0A 20 20 20 20 69 66 20 28 66 6B 3D 3D 4E 55 4C 4C 29 0A 20 20 20 20 7B 0A 20 20 20 20 20 20 20 20 70 72 69 6E 74 66 28 22 46 61 69 6C 65 64 20 74 6F 20 6F 70 65 6E 20 25 73 5C 6E 22 2C 20 6B 65 79 29 3B 0A 20 20 20 20 20 20 20 20 65 78 69 74 28 2D 31 29 3B 0A 20 20 20 20 7D 0A 20 20 20 20 0A 20 20 20 20 66 73 65 65 6B 28 66 70 2C 20 30 2C 20 53 45 45 4B 5F 45 4E 44 29 3B 0A 20 20 20 20 64 77 6F 72 64 20 6C 65 6E 67 74 68 20 3D 20 28 64 77 6F 72 64 29 66 74 65 6C 6C 28 66 70 29 3B 0A 20 20 20 20 66 73 65 65 6B 28 66 70 2C 20 30 2C 20 53 45 45 4B 5F 53 45 54 29 3B 0A 20 20 20 20 0A 20 20 20 20 66 77 72 69 74 65 28 26 6C 65 6E 67 74 68 2C 20 34 2C 20 31 2C 20 66 63 29 3B 0A 20 20 20 20 0A 20 20 20 20 66 6F 72 20 28 69 6E 74 20 69 3D 30 3B 20 69 3C 6C 65 6E 67 74 68 3B 20 69 2B 3D 34 29 0A 20 20 20 20 7B 0A 20 20 20 20 20 20 20 20 64 77 6F 72 64 20 70 3B 0A 20 20 20 20 20 20 20 20 66 72 65 61 64 28 26 70 2C 20 34 2C 20 31 2C 20 66 70 29 3B 0A 20 20 20 20 20 20 20 20 64 77 6F 72 64 20 6B 20 3D 20 67 65 6E 72 61 6E 64 5F 69 6E 74 33 32 28 29 3B 0A 20 20 20 20 20 20 20 20 64 77 6F 72 64 20 63 20 3D 20 70 5E 6B 3B 0A 20 20 20 20 20 20 20 20 0A 20 20 20 20 20 20 20 20 66 77 72 69 74 65 28 26 63 2C 20 34 2C 20 31 2C 20 66 63 29 3B 0A 20 20 20 20 20 20 20 20 66 77 72 69 74 65 28 26 6B 2C 20 34 2C 20 31 2C 20 66 6B 29 3B 0A 20 20 20 20 7D 0A 20 20 20 20 0A 20 20 20 20 66 63 6C 6F 73 65 28 66 70 29 3B 0A 20 20 20 20 66 63 6C 6F 73 65 28 66 63 29 3B 0A 20 20 20 20 66 63 6C 6F 73 65 28 66 6B 29 3B 0A 7D 0A 0A 76 6F 69 64 20 64 65 63 72 79 70 74 28 63 6F 6E 73 74 20 63 68 61 72 20 2A 70 6C 61 69 6E 2C 20 63 6F 6E 73 74 20 63 68 61 72 20 2A 63 72 79 70 74 2C 20 63 6F 6E 73 74 20 63 68 61 72 20 2A 6B 65 79 29 0A 7B 0A 20 20 20 20 46 49 4C 45 20 2A 66 70 20 3D 20 66 6F 70 65 6E 28 70 6C 61 69 6E 2C 20 22 77 62 22 29 3B 0A 20 20 20 20 69 66 20 28 66 70 3D 3D 4E 55 4C 4C 29 0A 20 20 20 20 7B 0A 20 20 20 20 20 20 20 20 70 72 69 6E 74 66 28 22 46 61 69 6C 65 64 20 74 6F 20 6F 70 65 6E 20 25 73 5C 6E 22 2C 20 70 6C 61 69 6E 29 3B 0A 20 20 20 20 20 20 20 20 65 78 69 74 28 2D 31 29 3B 0A 20 20 20 20 7D 0A 20 20 20 20 46 49 4C 45 20 2A 66 63 20 3D 20 66 6F 70 65 6E 28 63 72 79 70 74 2C 20 22 72 62 22 29 3B 0A 20 20 20 20 69 66 20 28 66 63 3D 3D 4E 55 4C 4C 29 0A 20 20 20 20 7B 0A 20 20 20 20 20 20 20 20 70 72 69 6E 74 66 28 22 46 61 69 6C 65 64 20 74 6F 20 6F 70 65 6E 20 25 73 5C 6E 22 2C 20 63 72 79 70 74 29 3B 0A 20 20 20 20 20 20 20 20 65 78 69 74 28 2D 31 29 3B 0A 20 20 20 20 7D 0A 20 20 20 20 46 49 4C 45 20 2A 66 6B 20 3D 20 66 6F 70 65 6E 28 6B 65 79 2C 20 22 72 62 22 29 3B 0A 20 20 20 20 69 66 20 28 66 6B 3D 3D 4E 55 4C 4C 29 0A 20 20 20 20 7B 0A 20 20 20 20 20 20 20 20 70 72 69 6E 74 66 28 22 46 61 69 6C 65 64 20 74 6F 20 6F 70 65 6E 20 25 73 5C 6E 22 2C 20 6B 65 79 29 3B 0A 20 20 20 20 20 20 20 20 65 78 69 74 28 2D 31 29 3B 0A 20 20 20 20 7D 0A 20 20 20 20 0A 20 20 20 20 64 77 6F 72 64 20 6C 65 6E 67 74 68 3B 0A 20 20 20 20 66 72 65 61 64 28 26 6C 65 6E 67 74 68 2C 20 34 2C 20 31 2C 20 66 63 29 3B 0A 20 20 20 20 0A 20 20 20 20 66 6F 72 20 28 69 6E 74 20 69 3D 30 3B 20 69 3C 6C 65 6E 67 74 68 3B 20 69 2B 3D 34 29 0A 20 20 20 20 7B 0A 20 20 20 20 20 20 20 20 64 77 6F 72 64 20 63 3B 0A 20 20 20 20 20 20 20 20 66 72 65 61 64 28 26 63 2C 20 34 2C 20 31 2C 20 66 63 29 3B 0A 20 20 20 20 20 20 20 20 64 77 6F 72 64 20 6B 3B 0A 20 20 20 20 20 20 20 20 66 72 65 61 64 28 26 6B 2C 20 34 2C 20 31 2C 20 66 6B 29 3B 0A 20 20 20 20 20 20 20 20 64 77 6F 72 64 20 70 20 3D 20 63 5E 6B 3B 0A 20 20 20 20 20 20 20 20 0A 20 20 20 20 20 20 20 20 66 77 72 69 74 65 28 26 70 2C 20 6D 69 6E 28 34 2C 6C 65 6E 67 74 68 2D 69 29 2C 20 31 2C 20 66 70 29 3B 0A 20 20 20 20 7D 0A 20 20 20 20 0A 20 20 20 20 66 63 6C 6F 73 65 28 66 70 29 3B 0A 20 20 20 20 66 63 6C 6F 73 65 28 66 63 29 3B 0A 20 20 20 20 66 63 6C 6F 73 65 28 66 6B 29 3B 0A 7D 0A";
let cpp_data: Vec<u8> = cpp_hex.split_whitespace()
    .map(|s| u8::from_str_radix(s, 16).unwrap())
    .collect();

// 2. encrypt.enc を読み込み
let mut enc_data = Vec::new();
File::open("encrypt/encrypt.enc").expect("enc file not found").read_to_end(&mut enc_data).unwrap();

// 3. 状態復元
let mut recovered_outputs = Vec::new();
for i in 0..624 {
    let c_chunk = u32::from_le_bytes([enc_data[i*4+4], enc_data[i*4+5], enc_data[i*4+6], enc_data[i*4+7]]);
    let p_chunk = u32::from_le_bytes([cpp_data[i*4], cpp_data[i*4+1], cpp_data[i*4+2], cpp_data[i*4+3]]);
    recovered_outputs.push(c_chunk ^ p_chunk);
}

let mut mt = MT19937::from_outputs(&recovered_outputs);

// 4. encrypt.cpp で消費された残りの乱数を読み飛ばす
// (ファイルサイズ 2600 / 4 = 650 回)
for _ in 0..26 { mt.genrand_int32(); }

// 5. flag.enc を復号
let mut flag_enc_data = Vec::new();
File::open("encrypt/flag.enc").expect("flag file not found").read_to_end(&mut flag_enc_data).unwrap();
let flag_size = u32::from_le_bytes([flag_enc_data[0], flag_enc_data[1], flag_enc_data[2], flag_enc_data[3]]) as usize;

let mut flag_dec_data = Vec::new();
for i in 0..((flag_size + 3) / 4) {
    let k = mt.genrand_int32();
    let c_idx = i * 4 + 4;
    let mut c_bytes = [0u8; 4];
    for j in 0..4 {
        if c_idx + j < flag_enc_data.len() { c_bytes[j] = flag_enc_data[c_idx + j]; }
    }
    let p_chunk = u32::from_le_bytes(c_bytes) ^ k;
    flag_dec_data.extend_from_slice(&p_chunk.to_le_bytes());
}

{
    let mut out_file = File::create("encrypt/flag.jpg").unwrap();
    out_file.write_all(&flag_dec_data[0..flag_size]).unwrap();
    // ここでブロックを抜けるときに自動でクローズ（ロック解放）されます
} 
println!("Success! 'encrypt/flag.jpg' has been released.");